In [16]:
# Paso 1: importar librerías
# ------------------------------------------------------------
# Para hacer "web scraping" (extraer información de una página web)
# necesitamos dos herramientas que trabajan en equipo:
#
# 1) requests      -> el "cartero": va a Internet y nos trae la página.
#                     Hace lo mismo que tu navegador cuando escribes una URL,
#                     pero en lugar de mostrarla bonita, nos da el código HTML.
#
# 2) BeautifulSoup -> el "traductor": el HTML es un texto larguísimo lleno de
#                     etiquetas (<p>, <span>, <div>...). BeautifulSoup lo
#                     organiza para que podamos buscar partes concretas fácilmente.
#
# Si no están instaladas:  pip install requests beautifulsoup4
import requests
from bs4 import BeautifulSoup

In [17]:
# Paso 2: ingresar un tema y construir el enlace de Wikipedia directamente
# ------------------------------------------------------------
# Las direcciones de Wikipedia en español siguen siempre el mismo patrón:
#     https://es.wikipedia.org/wiki/<Nombre_del_artículo>
# Ejemplo: "Inteligencia artificial" -> https://es.wikipedia.org/wiki/Inteligencia_artificial
# Así que solo tenemos que construir esa dirección a partir de lo que escriba el usuario.

# input() muestra un mensaje y espera a que el usuario escriba algo.
# Lo que escriba se guarda como texto (str) en la variable "tema".
tema = input("Introduce un tema: ")

# Limpiamos el texto antes de meterlo en la URL:
#   .strip()              -> quita los espacios sobrantes al principio y al final
#                            ("  Python  " -> "Python")
#   .replace(" ", "_")    -> cambia los espacios por guiones bajos, porque
#                            Wikipedia usa "_" en sus URLs en lugar de espacios
tema_url = tema.strip().replace(" ", "_")

# f-string: la "f" delante de las comillas permite meter variables entre {}.
# Python sustituye {tema_url} por su valor para formar el enlace completo.
link = f"https://es.wikipedia.org/wiki/{tema_url}"

# Mostramos el enlace para comprobar que se ha construido bien
print("Enlace construido:", link)

Enlace construido: https://es.wikipedia.org/wiki/Inteligencia_artificial


In [18]:
# Paso 3: hacer scraping de Wikipedia
# ------------------------------------------------------------
# Aquí ocurre la "magia": descargamos la página y la analizamos.

# Cabeceras (headers): es información que enviamos junto con la petición.
# El "User-Agent" le dice al servidor QUIÉN hace la petición.
# Algunos sitios rechazan las peticiones que no parecen venir de un navegador,
# así que nos presentamos como "Mozilla/5.0" (un navegador normal).
headers = {'User-Agent': 'Mozilla/5.0'}

# requests.get() pide la página a Internet (como pulsar Enter en el navegador).
# El resultado es un objeto "respuesta" que guardamos en "pagina". Contiene:
#   - pagina.status_code -> un número que indica si todo fue bien
#   - pagina.text        -> el código HTML completo de la página (un gran texto)
pagina = requests.get(link, headers=headers)

# Códigos de estado HTTP más habituales:
#   200 -> OK, la página existe y nos la han enviado
#   404 -> la página no existe (por ejemplo, el tema está mal escrito)
#   403 -> acceso prohibido
# Por eso comprobamos que sea 200 antes de continuar.
if pagina.status_code != 200:
    print("No se pudo acceder al artículo.")
else:
    # Creamos la "sopa": BeautifulSoup convierte el texto HTML en una
    # estructura ordenada (un árbol de etiquetas) en la que podemos buscar.
    # 'html.parser' es el analizador que viene incluido con Python.
    soup = BeautifulSoup(pagina.text, 'html.parser')

    # Buscamos el título del artículo.
    # Inspeccionando la página (clic derecho > "Inspeccionar" en el navegador)
    # se ve que Wikipedia escribe el título así:
    #     <span class="mw-page-title-main">Inteligencia artificial</span>
    # soup.find() devuelve el PRIMER elemento que cumple las condiciones:
    #   - 'span'                      -> el tipo de etiqueta
    #   - class_='mw-page-title-main' -> su clase CSS (se escribe "class_" con
    #                                    guion bajo porque "class" es una palabra
    #                                    reservada de Python)
    # .text extrae solo el texto que hay dentro de la etiqueta, sin el HTML.
    titulo = soup.find('span', class_='mw-page-title-main').text
    print("Título del artículo:", titulo)

Título del artículo: Inteligencia artificial


In [19]:
# Paso 4: extraer y mostrar contenido del artículo
# ------------------------------------------------------------
# En HTML, cada párrafo de texto va dentro de una etiqueta <p> ... </p>.
# Vamos a recoger todos esos párrafos y juntarlos en un solo texto.

# Empezamos con una cadena vacía donde iremos acumulando el texto
contenido = ""

# soup.find_all('p') devuelve una LISTA con TODAS las etiquetas <p> de la página
# (a diferencia de find(), que solo devuelve la primera).
# El bucle for recorre esa lista párrafo por párrafo:
for p in soup.find_all('p'):
    # p.text  -> el texto del párrafo, sin etiquetas HTML
    # + "\n"  -> añadimos un salto de línea para separar un párrafo del siguiente
    # +=      -> es un atajo de: contenido = contenido + ...
    contenido += p.text + "\n"

# El artículo puede ser muy largo, así que solo mostramos una muestra.
# contenido[:1000] es un "slicing": toma los caracteres desde el inicio
# hasta la posición 1000 (sin incluirla).
print("Contenido del artículo (primeros 1000 caracteres):\n")
print(contenido[:1000])

Contenido del artículo (primeros 1000 caracteres):

La inteligencia artificial, abreviada como IA o AI (por su nombre en inglés: artificial intelligence), en el contexto de las ciencias de la computación, es una disciplina y un conjunto de capacidades cognoscitivas e intelectuales expresadas por sistemas informáticos o combinaciones de algoritmos cuyo propósito es la creación de máquinas que imiten la inteligencia humana.[1]
Estas tecnologías permiten que las máquinas aprendan de la experiencia, se adapten a nuevas entradas y realicen tareas humanas como el reconocimiento de voz, la toma de decisiones, la traducción de idiomas o la visión por computadora.[2][3]
En la actualidad, la inteligencia artificial abarca una gran variedad de subcampos. Estos van desde áreas de propósito general, aprendizaje y percepción, a otras más específicas como el reconocimiento de voz, el juego de ajedrez, la demostración de teoremas matemáticos, el cálculo y la resolución de problemas aritméticos y algeb

In [20]:
# Guardar el texto completo en un archivo
# ------------------------------------------------------------
# Guardamos TODO el contenido (no solo los 1000 primeros caracteres)
# en un archivo .txt para poder consultarlo o analizarlo después.
nombre_archivo = "articulo_completo.txt"

# open() abre (o crea) un archivo:
#   - "w"              -> modo escritura (write). ¡Ojo! Si el archivo ya existe,
#                         se borra su contenido y se escribe de nuevo.
#   - encoding="utf-8" -> para que las tildes y la ñ se guarden correctamente
# "with" cierra el archivo automáticamente al terminar el bloque,
# aunque ocurra un error. Es la forma recomendada de trabajar con archivos.
with open(nombre_archivo, "w", encoding="utf-8") as archivo:
    archivo.write(contenido)

# El archivo se crea en la misma carpeta donde está este notebook
print(f"Texto completo guardado en: {nombre_archivo}")

Texto completo guardado en: articulo_completo.txt
